# Studying noise influence on the energy estimation

In [1]:
#Qiskit modules
import qiskit
from qiskit import QuantumRegister as Q_R
from qiskit import ClassicalRegister as C_R
from qiskit_aer import Aer
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
#from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService
from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer.noise import NoiseModel, depolarizing_error

#math modules
import math
import numpy as np
import random

#Showing results
import matplotlib.pyplot as plt

# SciPy minimizer routine
from scipy.optimize import minimize

import time
import sys
sys.path.insert(0, 'C:/Users/Oleg/Google Диск/QC/Codes/QC-qiskit-codes/Library')

#My libraries
import vqe_funcs
import aux_func as af

from vqe_funcs import sigma_x, sigma_y, sigma_z, kinetic_energy, full_ham, particles_number_control

#constants
PI = np.pi

In [4]:
def uniform_z(nodes_number):
    mag = []
    for i in range(nodes_number):
        mag.append([0, 0, 1])
    return mag

In [5]:
# Anzatz from Yordanov paper including only SINGLE QUBITs excitations. Initial state is FM state 
def anzatz_4(theta, nodes_number, electrons_number):
    qubits_number = 2 * nodes_number
    q_r = Q_R(qubits_number)
    v_qc = QuantumCircuit(q_r)
    #initial state preparation
    for i in range(electrons_number):
        v_qc.x(i)
    #Variable block
    v_qc = vqe_funcs.exc_yordanov_single_only_no_ladder(qubits_number, theta, v_qc, 0)
    return v_qc

In [41]:
nodes_number = 2 #number of nodes
electrons_number = 2

n_qubits = 2 * nodes_number
len_tot = int(n_qubits * (n_qubits - 1) / 2) #number of parameters

#Parameter space initialization
x0 = []
for i in range(len_tot):
    x0.append(random.random() * 0.01)

q_r = Q_R(nodes_number * 2)
cl_r = C_R(nodes_number * 2)
qc_f = QuantumCircuit(q_r,cl_r)
qc_1 = anzatz_4(x0, nodes_number, electrons_number)
qc_f.append(qc_1, q_r)
qc_f.measure_all()


SimulatorAer = AerSimulator()
circ = transpile(qc_f, backend = SimulatorAer)
result = SimulatorAer.run(circ,shots = 1).result()
count_noiseless = result.get_counts()

In [42]:
count_noiseless

{'0011 0000': 1}

In [43]:
#qc_f.decompose().draw('mpl')

In [46]:
from qiskit_aer.primitives import SamplerV2 as Sampler
noise_model = NoiseModel()
cx_depolarizing_prob = 0.01
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)
noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)

# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(qc_f)
pub = (isa_circuit)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'0011': 909,
 '0111': 17,
 '0001': 16,
 '0110': 6,
 '1111': 6,
 '1011': 23,
 '0000': 11,
 '0010': 15,
 '1010': 7,
 '0101': 6,
 '1001': 8}

In [37]:
pub_result.data

DataBin(c3=BitArray(<shape=(), num_shots=1024, num_bits=4>))

In [50]:
# Energy estimation for each pure state
q_r = Q_R(nodes_number * 2)
cl_r = C_R(nodes_number * 2)
qc_f = QuantumCircuit(q_r,cl_r)

n_states = pow(2, nodes_number * 2)

qc_f.x(0)
nodes_number = 3 #number of nodes
electrons_number = 3
J = 0.2 #s-d exchange constant
t = 1 #hopping matrix element (kinetic energy coefficient)
U_c = 10

mag = []
mag = uniform_z(nodes_number)

#define the Hamiltonian
hamiltonian = full_ham(nodes_number, mag, J, t, U_c, periodic = False)

estimator = StatevectorEstimator()
job = estimator.run([(qc_f, hamiltonian)])
estimator_expvals = job.result()[0].data.evs

In [51]:
estimator_expvals

array(5.2)

'0b10'